# ISBP Non-Regular — Exp 1B: No Weaker VEST Cut (w̄ ∈ {4,...,15})

Same as Experiment 1 but with **weaker VEST cut turned off** (`add_vest_cut=False`).
This measures the effectiveness of the weaker VEST cut on solver performance
for non-regular (irregular) instances.

> **Important**: Run this notebook in a **fresh kernel** (shut down notebook 03 first)
> to ensure independent execution environment and eliminate memory/ordering bias.

In [ ]:
from pathlib import Path
import json
import time
import csv
import math
import traceback

import numpy as np
import pandas as pd

from gipg.isbp.instance import ISBPInstance
from gipg.isbp.heuristics import (
    brd_random_restart, alpha_of_profile,
    is_regular_instance,
)
from gipg.isbp.gzr import solve_gzr
from gipg.isbp.objectives import profile_costs
from gipg.isbp.social_optimum import solve_social_optimum, compute_pos

OUTDIR = Path('outputs')
OUTDIR.mkdir(exist_ok=True)
print('Imports OK.')

## 1. Non-Regular Instance Generation

In [ ]:
def generate_nonregular_instance(
    n_players: int,
    m_bins: int,
    w_bar: int,
    u_bar: int,
    delta: float = 0.2,
    seed: int = 0,
) -> ISBPInstance:
    """
    Generate a non-regular ISBP instance with heterogeneous parameters.
    """
    rng = np.random.default_rng(seed)
    players = list(range(n_players))
    bins = list(range(m_bins))

    w_lo = max(1, math.ceil(w_bar * (1 - delta)))
    w_hi = math.floor(w_bar * (1 + delta))
    u_lo = max(1, math.ceil(u_bar * (1 - delta)))
    u_hi = math.floor(u_bar * (1 + delta))
    c_lo = max(1, math.ceil(u_bar * (1 - delta)))
    c_hi = math.floor(u_bar * (1 + delta))

    weights = {i: int(rng.integers(w_lo, w_hi + 1)) for i in players}
    capacities = {j: int(rng.integers(u_lo, u_hi + 1)) for j in bins}
    costs = {j: float(rng.integers(c_lo, c_hi + 1)) for j in bins}

    total_w = sum(weights.values())
    total_u = sum(capacities.values())
    if total_u < total_w:
        deficit = total_w - total_u
        sorted_bins = sorted(bins, key=lambda j: capacities[j])
        idx = 0
        while deficit > 0:
            j = sorted_bins[idx % m_bins]
            capacities[j] += 1
            deficit -= 1
            idx += 1

    T = {j: list(range(0, capacities[j] + 1)) for j in bins}
    S = {
        i: {
            j: {
                t: list(range(0, min(t, weights[i]) + 1))
                for t in T[j]
            }
            for j in bins
        }
        for i in players
    }
    return ISBPInstance(players, bins, costs, capacities, weights, T, S)

print('generate_nonregular_instance defined.')

In [ ]:
def enumerate_tuples_regular(n_range, m_range, w_range):
    tuples = []
    for n in n_range:
        for m in range(n + m_range[0], n + m_range[1]):
            if m <= 0:
                continue
            for w in w_range:
                nw = n * w
                q, r = divmod(nw, m)
                if r == 0:
                    u_candidates = [q, q+1, q+2, q+3]
                else:
                    u_candidates = [q+1, q+2, q+3, q+4]
                for u in u_candidates:
                    if (m*u - nw >= 0) and (m <= math.ceil(nw/u)):
                        tuples.append((n, m, w, u))
    return tuples

n_range = range(4, 11)
m_range = [-2, 4]
w_range = range(4, 16)
DELTA = 0.2

tuples = enumerate_tuples_regular(n_range, m_range, w_range)
print(f'Base tuples: {len(tuples)}')

In [ ]:
SEEDS = [0]

# Controls
TOTAL_TIME_LIMIT = 600.0
SO_TIME_LIMIT = 600.0

# Output CSV path
exp1b_path = OUTDIR / 'isbp_nonreg_exp1b_no_vest.csv'
err_path = OUTDIR / 'isbp_nonreg_errors.csv'

# CSV append helper
def append_row(path: Path, header, rowdict):
    new_file = not path.exists()
    with path.open('a', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=header)
        if new_file:
            w.writeheader()
        w.writerow({k: rowdict.get(k, '') for k in header})

print(f'Tuples: {len(tuples)}, Seeds: {len(SEEDS)}, Total runs: {len(tuples)*len(SEEDS)}')

## 2. Experiment 1B: BRD + ZR without Weaker VEST Cut

Same as Experiment 1 but with **weaker VEST cut turned off** (`add_vest_cut=False`).
For each instance:
1. **BRD** warm-starts **ZR @ alpha=1** (if BRD found a PNE)
2. **ZR** with `add_vest_cut=False`, `stop_at_first_pne=False`
3. **Social Optimum** computed separately
4. **POS** = best_PNE_cost / SO_cost

In [ ]:
exp1b_header = [
    'seed', 'n', 'm', 'w_bar', 'u_bar', 'delta', 'regular',
    'sum_w', 'sum_u', 'feasibility_repairs',
    'brd_found_pne', 'brd_alpha', 'brd_time',
    'zr_status', 'zr_gurobi_status', 'zr_mip_gap', 'zr_obj_bound', 'zr_runtime', 'zr_time',
    'zr_eis_added_total', 'zr_eis_added_symmetric', 'zr_eis_added_core',
    'zr_first_pne_time',
    'best_pne_cost',
    'so_status', 'so_cost', 'so_time',
    'pos',
]

err_header = ['seed', 'n', 'm', 'w_bar', 'u_bar', 'delta', 'stage', 'error', 'traceback']

results_exp1b = []

total = 0
for (n, m, w_bar, u_bar) in tuples:
    for seed in SEEDS:
        total += 1
        tag = f'n{n}_m{m}_wb{w_bar}_ub{u_bar}_d{DELTA}_s{seed}'

        row = {
            'seed': seed, 'n': n, 'm': m,
            'w_bar': w_bar, 'u_bar': u_bar, 'delta': DELTA,
        }

        try:
            inst = generate_nonregular_instance(
                n_players=n, m_bins=m,
                w_bar=w_bar, u_bar=u_bar,
                delta=DELTA, seed=seed,
            )
            regular = is_regular_instance(inst)
            row['regular'] = regular

            sum_w = sum(inst.weights.values())
            sum_u = sum(inst.capacities.values())
            u_lo = max(1, math.ceil(u_bar * (1 - DELTA)))
            u_hi = math.floor(u_bar * (1 + DELTA))
            nominal_sum_u = sum(
                int(np.random.default_rng(seed).integers(u_lo, u_hi + 1))
                for _ in range(m)
            )
            row['sum_w'] = sum_w
            row['sum_u'] = sum_u
            row['feasibility_repairs'] = max(0, sum_w - nominal_sum_u)

            # --- Phase 1: BRD ---
            x_brd, brd_pne, brd_time = brd_random_restart(
                inst, max_init=3, max_round=15,
                seed=seed,
            )
            brd_alpha = alpha_of_profile(inst, x_brd) if x_brd else float('inf')
            row['brd_found_pne'] = brd_pne
            row['brd_alpha'] = brd_alpha
            row['brd_time'] = brd_time

            # --- Phase 2: ZR @ alpha=1, weaker VEST cut OFF ---
            zr_time_limit = max(TOTAL_TIME_LIMIT - brd_time, 60.0)
            warm = x_brd if brd_pne else None

            zr = solve_gzr(
                inst,
                alpha=1.0,
                time_limit=zr_time_limit,
                log_to_console=0,
                stop_at_first_pne=False,
                warm_start=warm,
                add_vest_cut=False,   # <-- weaker VEST cut OFF
            )

            zr_status = zr.get('status')
            row['zr_gurobi_status'] = zr.get('gurobi_status')
            row['zr_mip_gap'] = zr.get('mip_gap')
            row['zr_obj_bound'] = zr.get('obj_bound')
            row['zr_runtime'] = zr.get('runtime')
            row['zr_time'] = brd_time + float(zr.get('runtime', 0))
            row['zr_eis_added_total'] = zr.get('eis_added_total', 0)
            row['zr_eis_added_symmetric'] = zr.get('eis_added_symmetric', 0)
            row['zr_eis_added_core'] = zr.get('eis_added_core', 0)
            row['zr_first_pne_time'] = (brd_time + zr['first_pne_time']
                                                    if zr.get('first_pne_time') is not None else None)

            # Post-ZR fallback
            zr_profile = zr.get('x_profile')
            if zr_profile is None and zr_status in ('INTERRUPTED', 'UNKNOWN') and brd_pne:
                verify_alpha = alpha_of_profile(inst, x_brd)
                if verify_alpha <= 1.0 + 1e-9:
                    zr_status = 'OPTIMAL'
                    zr_profile = x_brd

            row['zr_status'] = zr_status

            # Best PNE cost
            if zr_profile is not None:
                _, total_cost = profile_costs(
                    players=inst.players, bins=inst.bins,
                    costs=inst.costs, x_profile=zr_profile,
                )
                row['best_pne_cost'] = total_cost
            elif brd_pne and x_brd is not None:
                _, total_cost = profile_costs(
                    players=inst.players, bins=inst.bins,
                    costs=inst.costs, x_profile=x_brd,
                )
                row['best_pne_cost'] = total_cost
            else:
                row['best_pne_cost'] = None

            # --- Phase 3: Social Optimum ---
            so_res = solve_social_optimum(inst, time_limit=SO_TIME_LIMIT, verbose=False)
            row['so_status'] = so_res.status
            row['so_cost'] = so_res.opt_cost
            row['so_time'] = so_res.runtime

            # POS
            if row.get('best_pne_cost') is not None and so_res.opt_cost is not None and so_res.opt_cost > 0:
                row['pos'] = compute_pos(so_res.opt_cost, row['best_pne_cost'])
            else:
                row['pos'] = None

            results_exp1b.append(row)
            append_row(exp1b_path, exp1b_header, row)

            # Progress
            if total % 50 == 0 or total == len(tuples) * len(SEEDS):
                pne_str = 'BRD-PNE' if brd_pne else 'no-PNE'
                zr_str = f'{row["zr_status"]}'
                zr_t = f'{row["zr_time"]:.1f}s' if row.get('zr_time') is not None else '?s'
                pos_str = f'POS={row["pos"]:.3f}' if row.get('pos') is not None else 'POS=N/A'
                fpne_str = f'1stPNE={row["zr_first_pne_time"]:.1f}s' if row.get('zr_first_pne_time') is not None else '1stPNE=N/A'
                print(f'[{total}] {tag}: {pne_str} | ZR {zr_str} ({zr_t}) | {pos_str} | {fpne_str}')

        except Exception as e:
            append_row(err_path, err_header, {
                'seed': seed, 'n': n, 'm': m, 'w_bar': w_bar, 'u_bar': u_bar,
                'delta': DELTA, 'stage': 'EXP1B', 'error': repr(e),
                'traceback': traceback.format_exc(),
            })
            print(f'  !! EXP1B failed ({tag}): {e}')
            continue

df_exp1b = pd.DataFrame(results_exp1b)
print(f'\nExp1B complete: {len(df_exp1b)} rows saved to {exp1b_path}')

In [ ]:
# Experiment 1B Summary
print('=== Experiment 1B Summary (Non-Regular, No Weaker VEST Cut) ===')

n_total_1b = len(df_exp1b)
n_brd_pne_1b = df_exp1b['brd_found_pne'].sum()
n_regular_1b = df_exp1b['regular'].sum()
n_zr_found_1b = (df_exp1b['zr_status'] == 'OPTIMAL').sum()
n_zr_inf_1b = (df_exp1b['zr_status'] == 'INFEASIBLE').sum()
n_zr_tl_1b = (df_exp1b['zr_status'] == 'TIME_LIMIT').sum()

print(f'Total instances: {n_total_1b}')
print(f'Actually regular: {n_regular_1b}')
print(f'BRD found PNE: {n_brd_pne_1b} ({n_brd_pne_1b/n_total_1b:.1%})')
print(f'ZR OPTIMAL: {n_zr_found_1b} ({n_zr_found_1b/n_total_1b:.1%})')
print(f'ZR INFEASIBLE: {n_zr_inf_1b} ({n_zr_inf_1b/n_total_1b:.1%})')
print(f'ZR TIME_LIMIT: {n_zr_tl_1b} ({n_zr_tl_1b/n_total_1b:.1%})')

# By (n, m)
print('\n--- By (n, m) ---')
summary = df_exp1b.groupby(['n', 'm']).agg(
    count=('seed', 'count'),
    brd_pne_rate=('brd_found_pne', 'mean'),
    zr_opt_rate=('zr_status', lambda x: (x == 'OPTIMAL').mean()),
    avg_zr_time=('zr_time', 'mean'),
    avg_pos=('pos', 'mean'),
).round(4)
print(summary)